# 01 — Quickstart: Proxy a tool through STM

This notebook walks you through the minimum viable memtomem-stm
setup: register one upstream MCP server, talk to STM as an MCP
client, and read the proxy stats.

**You will learn:**

- How to register an upstream MCP server with the `mms` CLI
- How STM exposes proxied tools (namespaced with a prefix)
- How to call tools via the MCP Python client
- How to read `stm_proxy_stats` to see what STM did

**Prereqs:** `uv sync` (installs the dev group including Jupyter).
No external services required — we use a trivial echo server
shipped under `_fixtures/`.

**No state is leaked.** Every notebook isolates its proxy config,
cache, and metrics into a temp directory via environment
variables. Your real `~/.memtomem/` is untouched.

## 1. Isolate state and import helpers

In [ ]:
# Add notebooks/ to sys.path so `_helpers` is importable regardless of
# where Jupyter was launched (from the repo root or from notebooks/).
import sys
from pathlib import Path

_cwd = Path.cwd()
if (_cwd / "_helpers.py").exists():
    sys.path.insert(0, str(_cwd))
elif (_cwd / "notebooks" / "_helpers.py").exists():
    sys.path.insert(0, str(_cwd / "notebooks"))
else:
    raise RuntimeError(
        "Cannot find notebooks/_helpers.py — run Jupyter from the repo "
        "root or from the notebooks/ directory."
    )

In [ ]:
import subprocess

from _helpers import (
    isolate_stm_state,
    stm_session,
    extract_text,
    fixtures_dir,
)

config_path = isolate_stm_state(prefix="nb01_")
print(f"STM config → {config_path}")
print(f"Fixtures   → {fixtures_dir()}")

## 2. Register an upstream MCP server

We'll point STM at a tiny echo server that ships with the
repo. In production you would use `--command npx --args
"-y @modelcontextprotocol/server-filesystem /your/path"` or
any other MCP server — the mechanics are identical.

In [ ]:
echo_script = fixtures_dir() / "echo_mcp.py"
result = subprocess.run(
    [
        "uv", "run", "mms", "add", "echo",
        "--config", str(config_path),
        "--command", "uv",
        "--args", f"run python {echo_script}",
        "--prefix", "echo",
    ],
    capture_output=True, text=True, check=True,
)
print(result.stdout.strip())

In [ ]:
# Show what mms wrote to the config file
print(config_path.read_text())

## 3. Connect to STM as an MCP client

`stm_session()` spawns `memtomem-stm` as a subprocess using
the stdio transport, wraps it in an MCP `ClientSession`, and
initializes the connection. This is exactly what Claude Code
or Cursor do under the hood.

> **Heads-up:** STM logs its lifecycle to stderr — those
> `INFO` lines you see interleaved below are normal and show
> the proxy manager connecting to the echo upstream.

In [ ]:
async with stm_session() as session:
    tools_response = await session.list_tools()
    tool_names = sorted(t.name for t in tools_response.tools)
    print(f"STM exposes {len(tool_names)} tools:")
    for name in tool_names:
        print(f"  {name}")

Two groups of tools appear above:

- **`echo__*`** — the upstream echo server's tools, namespaced
  with the `--prefix echo` you passed to `mms add`. STM
  dynamically discovers these when it starts.
- **`stm_proxy_*` / `stm_surfacing_*`** — STM's own built-in
  control tools (stats, health, selective chunk retrieval,
  progressive read, cache clear, surfacing feedback).

## 4. Call the proxied tool

In [ ]:
async with stm_session() as session:
    result = await session.call_tool("echo__say", {"text": "hello from the notebook"})
    print(extract_text(result))

## 5. Read the proxy stats

In [ ]:
async with stm_session() as session:
    # Trigger one real call so the stats show non-zero activity
    await session.call_tool("echo__say", {"text": "counting this one"})
    stats = await session.call_tool("stm_proxy_stats", {})
    print(extract_text(stats))

## Recap

You just routed a tool call through STM:

1. `mms add` wrote an entry in an isolated `stm_proxy.json`
2. STM loaded that config on startup and spawned `echo_mcp.py`
   as a subprocess
3. Your `session.call_tool("echo__say", ...)` call flowed
   `notebook → STM → echo → STM → notebook`
4. STM's `TokenTracker` recorded the call in its stats

The echo server returns tiny responses, so no compression fires
yet. **Notebook 02** swaps in a large structured document and
shows STM's selective compression producing a table of contents.